# 位置编码：正余弦 PE vs RoPE

## 对比总结

| 维度 | 正余弦位置编码 (Sinusoidal PE) | RoPE (Rotary Position Embedding) |
|------|-------------------------------|----------------------------------|
| **类型** | 绝对位置编码 | 相对位置编码 |
| **作用位置** | 加在 embedding 上 (输入层) | 作用在 Q 和 K 上 (注意力层) |
| **操作方式** | 加法: x' = x + PE(pos) | 旋转(乘法): Q' = R(pos) * Q |
| **位置信息** | 编码绝对位置 ("我在位置3") | 编码相对位置 ("我们相距2") |
| **核心性质** | 不同位置编码正交 | Q_m · K_n 只依赖 (m-n) |
| **外推性** | 差 (超过训练长度失效) | 较好 (配合NTK外推更佳) |
| **是否可学习** | 否 (固定) | 否 (固定) |
| **对语义的影响** | 加法改变模长和方向 | 旋转保持模长不变 |
| **代表模型** | 原始Transformer, BERT(可学习版) | LLaMA, GPT-NeoX, PaLM |
| **计算开销** | 预计算一次，几乎无开销 | 每层Q/K都要旋转，略多 |

### 正余弦 PE 的优缺点

**优点:**
- 实现简单，无需训练参数
- 不同频率的sin/cos波能编码多尺度位置信息
- 理论上任意两个位置的PE可通过线性变换互相转换

**缺点:**
- 绝对位置编码，不能直接建模相对位置关系
- 加法形式会改变词向量的模长和方向，可能干扰语义
- 外推性差，超出训练长度后效果显著下降

### RoPE 的优缺点

**优点:**
- 天然的相对位置编码: 内积只依赖相对距离
- 旋转变换保持向量模长不变，不破坏语义空间
- 外推性好，配合NTK-aware插值可扩展到很长序列
- 无需额外参数

**缺点:**
- 实现略复杂
- 每一层的Q/K都需要做旋转，计算量略大于加法PE
- 高维时远距离位置的区分度可能下降

## 为什么 RoPE 比正余弦 PE 更优雅？—— Attention Score 展开对比

### 正余弦 PE 的 Attention Score

正余弦PE通过加法注入位置信息：$Q = W_q(X_m + P_m)$，$K = W_k(X_n + P_n)$

简化掉 $W_q, W_k$（不影响分析），Attention Score 展开为：

$$\text{Score} = (X_m + P_m)(X_n + P_n)^T = \underbrace{X_m X_n^T}_{\text{纯语义}} + \underbrace{X_m P_n^T + P_m X_n^T}_{\text{交叉噪声}} + \underbrace{P_m P_n^T}_{\text{纯位置}}$$

**四项的含义：**

| 项 | 含义 | 问题 |
|---|------|------|
| $X_m X_n^T$ | 纯语义相似度 | 这是我们想要的 |
| $X_m P_n^T$ | 位置m的**内容**与位置n的**位置编码**交互 | 噪声：词义不应受对方位置影响 |
| $P_m X_n^T$ | 位置m的**位置编码**与位置n的**内容**交互 | 噪声：同上 |
| $P_m P_n^T$ | 纯位置关系 | 有用，但只依赖绝对位置 |

中间两项 $X_m P_n^T$ 和 $P_m X_n^T$ 是**交叉噪声**——它们把语义和位置耦合在一起，没有明确的语义。

### RoPE 的 Attention Score

RoPE 通过旋转注入位置信息：$Q = R_m X_m$，$K = R_n X_n$

$$\text{Score} = (R_m X_m)(R_n X_n)^T = R_m X_m X_n^T R_n^T = X_m \underbrace{R_{n-m}}_{\text{相对位置}} X_n^T$$

（利用旋转矩阵的正交性 $R_m R_n^T = R_{m-n}$）

**只有一项！** 位置信息 $R_{n-m}$ 作为调制因子优雅地嵌入语义内积中，且只依赖**相对位置** $(n-m)$。

### 对比

| | 正余弦 PE | RoPE |
|---|-----------|------|
| Score 展开项数 | 4 项 | 1 项 |
| 交叉噪声 | 有 ($X_m P_n^T + P_m X_n^T$) | 无 |
| 位置信息 | 绝对位置 $P_m, P_n$ | 相对位置 $R_{n-m}$ |
| 语义与位置 | 耦合（加法混在一起） | 解耦（旋转不改变模长） |

In [ ]:
import torch
import torch.nn as nn
import math

## 正余弦位置编码 (Sinusoidal Positional Encoding)

### 公式

$$PE(pos, 2i) = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

$$PE(pos, 2i+1) = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

等价写法（用 exp-log 技巧避免大数）:

$$\text{div\_term} = e^{2i \cdot (-\ln 10000 / d_{model})} = \frac{1}{10000^{2i/d_{model}}}$$

$$PE(pos, 2i) = \sin(pos \cdot \text{div\_term})$$

$$PE(pos, 2i+1) = \cos(pos \cdot \text{div\_term})$$

### 使用方式

$$\text{output} = \text{embedding}(x) + PE[:\text{seq\_len}, :]$$

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    """正余弦位置编码"""

    def __init__(self, d_model, max_len=5000):
        super().__init__()
        # 创建位置编码矩阵: (max_len, d_model)
        pe = torch.zeros(max_len, d_model)

        # pos: (max_len, 1)
        pos = torch.arange(0, max_len).unsqueeze(1).float()

        # div_term: (d_model/2,)
        # = exp(2i * (-log(10000) / d_model)) = 1/10000^(2i/d_model)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        # 偶数维度用sin，奇数维度用cos
        pe[:, 0::2] = torch.sin(pos * div_term)  # (max_len, d_model/2)
        pe[:, 1::2] = torch.cos(pos * div_term)  # (max_len, d_model/2)

        # 注册为 buffer（不参与梯度更新，但会保存到模型中）
        # 加一个 batch 维度: (1, max_len, d_model)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        """x: (batch_size, seq_len, d_model)"""
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len, :]  # 加法注入位置信息

## RoPE (Rotary Position Embedding)

### 核心思想

将位置信息通过**旋转变换**编码到 Q 和 K 中，使得内积只依赖相对位置。

### 公式

对于维度 $d$ 的向量，两两配对 $(q_{2i}, q_{2i+1})$，做二维旋转:

$$q'_{2i} = q_{2i} \cdot \cos(m \cdot \theta_i) - q_{2i+1} \cdot \sin(m \cdot \theta_i)$$

$$q'_{2i+1} = q_{2i} \cdot \sin(m \cdot \theta_i) + q_{2i+1} \cdot \cos(m \cdot \theta_i)$$

其中:
- $m$ = 位置索引
- $\theta_i = \frac{1}{10000^{2i/d}}$ （和正余弦PE用相同的频率）

### 关键性质

$$\langle R(m)\mathbf{q},\; R(n)\mathbf{k} \rangle = \langle R(m-n)\mathbf{q},\; \mathbf{k} \rangle$$

旋转后的内积只依赖相对位置 $(m-n)$！

### 使用方式

```python
Q = W_q(x)              # 线性投影
K = W_k(x)
Q = apply_rope(Q, pos)  # 旋转
K = apply_rope(K, pos)
attn = Q @ K.T          # 内积天然包含相对位置信息
```

In [ ]:
class RotaryPositionalEncoding(nn.Module):
    """RoPE 旋转位置编码"""

    def __init__(self, d_model, max_len=5000, base=10000):
        super().__init__()
        # theta_i = 1/base^(2i/d), i=0,1,...,d/2-1
        # 用 exp-log 计算: exp(-2i/d * log(base))
        inv_freq = 1.0 / (base ** (torch.arange(0, d_model, 2).float() / d_model))
        self.register_buffer('inv_freq', inv_freq)  # (d_model/2,)

    def forward(self, x, seq_len=None):
        """
        x: (batch, seq_len, num_heads, head_dim) 或 (batch, num_heads, seq_len, head_dim)
        这里假设输入是 (batch, seq_len, head_dim)，简化版本
        返回旋转后的 x
        """
        if seq_len is None:
            seq_len = x.size(1)

        # positions: (seq_len,)
        positions = torch.arange(seq_len, device=x.device).float()

        # freqs: (seq_len, d/2) = outer product of positions and inv_freq
        freqs = torch.outer(positions, self.inv_freq)  # (seq_len, d/2)

        # cos/sin: (seq_len, d/2)
        cos = freqs.cos()
        sin = freqs.sin()

        return self._apply_rotary(x, cos, sin)

    def _apply_rotary(self, x, cos, sin):
        """应用旋转变换"""
        # x: (..., d)  把最后一维拆成两半
        d = x.shape[-1]
        x1 = x[..., :d//2]  # 前半部分
        x2 = x[..., d//2:]  # 后半部分

        # 旋转公式:
        # x1' = x1 * cos - x2 * sin
        # x2' = x1 * sin + x2 * cos
        rotated = torch.cat([
            x1 * cos - x2 * sin,
            x1 * sin + x2 * cos,
        ], dim=-1)

        return rotated

## 测试

In [ ]:
d_model = 64
seq_len = 10
batch_size = 2

x = torch.randn(batch_size, seq_len, d_model)

# === 正余弦 PE ===
sinusoidal_pe = SinusoidalPositionalEncoding(d_model)
out_sin = sinusoidal_pe(x)
print('=== 正余弦 PE ===')
print(f'输入:  {x.shape}')
print(f'输出:  {out_sin.shape}')
print(f'PE矩阵: {sinusoidal_pe.pe.shape}')
print(f'操作: x + PE (加法)')
print()

# === RoPE ===
rope = RotaryPositionalEncoding(d_model)
out_rope = rope(x)
print('=== RoPE ===')
print(f'输入:  {x.shape}')
print(f'输出:  {out_rope.shape}')
print(f'操作: 旋转变换 (乘法)')

## 验证 RoPE 的相对位置性质

核心性质: `<R(m)q, R(n)k>` 只依赖 `(m-n)`

In [ ]:
rope = RotaryPositionalEncoding(d_model)
q = torch.randn(1, 20, d_model)  # 一个长度20的序列
k = torch.randn(1, 20, d_model)

q_rot = rope(q)
k_rot = rope(k)

# 验证: <R(5)q, R(3)k> 应该等于 <R(7)q, R(5)k> (都相距2)
dot_5_3 = (q_rot[0, 5] * k_rot[0, 3]).sum().item()
dot_7_5 = (q_rot[0, 7] * k_rot[0, 5]).sum().item()
dot_10_8 = (q_rot[0, 10] * k_rot[0, 8]).sum().item()

print('验证相对位置性质 (相同q,k向量在不同绝对位置但相同相对距离=2):')
print(f'  注意: q,k在不同位置的值不同，所以上面的值不会相等')
print(f'  但如果q,k在所有位置相同:')
print()

# 用相同的q,k向量测试
q_same = torch.randn(1, 1, d_model).expand(1, 20, d_model).clone()
k_same = torch.randn(1, 1, d_model).expand(1, 20, d_model).clone()
q_rot2 = rope(q_same)
k_rot2 = rope(k_same)

dot_a = (q_rot2[0, 5] * k_rot2[0, 3]).sum().item()   # 距离=2
dot_b = (q_rot2[0, 7] * k_rot2[0, 5]).sum().item()   # 距离=2
dot_c = (q_rot2[0, 10] * k_rot2[0, 8]).sum().item()  # 距离=2
dot_d = (q_rot2[0, 5] * k_rot2[0, 2]).sum().item()   # 距离=3

print('相同向量不同位置，相对距离=2:')
print(f'  pos(5,3): {dot_a:.6f}')
print(f'  pos(7,5): {dot_b:.6f}')
print(f'  pos(10,8): {dot_c:.6f}')
print(f'  -> 三个值应该相等!')
print(f'\n相对距离=3:')
print(f'  pos(5,2): {dot_d:.6f}')
print(f'  -> 距离不同，值也不同')

## 验证 RoPE 保持模长不变

In [ ]:
q = torch.randn(1, 5, d_model)
q_rot = rope(q)

for i in range(5):
    norm_before = q[0, i].norm().item()
    norm_after = q_rot[0, i].norm().item()
    print(f'位置{i}: 旋转前模长={norm_before:.4f}, 旋转后模长={norm_after:.4f}, 差={abs(norm_before-norm_after):.6f}')
print('\n-> 旋转变换保持模长不变! (正余弦PE的加法则会改变模长)')

## 常见面试追问

| 问题 | 要点 |
|------|------|
| 为什么要位置编码? | Transformer的attention是置换不变的，不加位置编码则无法区分顺序 |
| sin/cos PE为什么能编码位置? | 不同频率的波组合，类似傅里叶级数，可唯一表示每个位置 |
| 为什么用sin和cos交替? | 任意两个位置的PE可通过线性变换互相转换(旋转矩阵) |
| RoPE为什么只作用在Q和K上? | 因为位置信息通过QK内积传递，V不需要位置信息 |
| RoPE如何实现相对位置? | 旋转矩阵的正交性: R(m)^T R(n) = R(n-m) |
| RoPE的外推怎么做? | NTK-aware插值: 修改base频率，或位置插值(PI) |
| 可学习PE(BERT)和固定PE区别? | 可学习PE效果略好但无法外推，固定PE可泛化到更长序列 |
| ALiBi是什么? | 不加PE，直接在attention score上加距离偏置，外推性最好 |